# EEG seizure prediction — waveforms → deep CNN + LSTM

This single notebook contains all project code. It reads **EEG only**, removes configured frequency noise, keeps the waveform samples, and trains a deep convolutional/LSTM network. There are no shallow classifiers, feature tables, external source-code modules or audit folders.

**Manual use:** change section 2, then press **Shift+Enter** cell by cell. A definition cell (`def` / `class`) teaches Python an operation; the following **Run** cell calls it. Read each table before continuing. Restart the kernel and run from the top for a new experiment.

Every run of **Start a session** creates a separate folder under `intermediate/`. It has exactly three subfolders: **processed**, **model**, **report**. Completed sessions are not overwritten and processing is not shared between sessions. For preparation only, set `RUN_TRAINING=False`.

**Input to the model:** a batch shaped **examples × selected EEG channels × waveform samples**. Filtering and optional downsampling alter the waveform, but no mean/power/entropy summary replaces it. The CNN learns local waveform patterns; the LSTM models the order of those learned patterns **within each observed input window**. It does not silently join separate recordings or claim hours of recurrent memory.

The forecast remains **30–40 minutes ahead** by default. Deep learning is an alternative to test, not a guarantee of better results. Patient separation, annotation coverage and the amount of usable seizure/background data still matter. No personalized baseline adaptation is implemented.

## 1. Load the Python libraries

**Output:** the selected Python and PyTorch versions. No data are read here. Install the packages using the one-time command in README if an import fails. In Jupyter, choose **Python (seizure_prediction)**.

In [ ]:
import copy
import hashlib
import importlib.metadata
import json
import itertools
import math
from pathlib import Path
import random
import re
import shutil
import sys
import time
import traceback
from datetime import datetime, timezone
from fractions import Fraction
from uuid import uuid4

import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.signal import butter, sosfilt, iirnotch, tf2sos, resample_poly, welch
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    brier_score_loss,
    confusion_matrix,
    precision_recall_curve,
)
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

print("Python:", sys.executable)
print("PyTorch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())


## 2. Settings — edit this cell

`SOURCE_DIR` contains original participant folders, directly: `source/sub-001/ses-01/eeg/...`. `SUBJECTS=None` discovers all people, including future downloads. `RECORDINGS` optionally selects exact full recording IDs printed by the inventory.

### EEG only: which files are opened?

The source-data discovery pattern is **`sub-*/ses-*/eeg/*_eeg.edf`**. The notebook opens these EEG EDFs, their EEG JSON metadata, and the matching seizure-event TSVs. It does **not** open ECG, EMG or movement files. Keeping those originals on disk does not load them into this pipeline's RAM or spend processing/training compute on them. The annotation TSV is needed to create seizure labels; it is not another sensor input.

### Where the EEG channels are recorded

An EEG **channel measures a voltage difference between electrodes**, rather than one point on the head. These are wearable **behind-the-ear** recordings, not in-ear electrodes or a selectable full-scalp 10-20 montage.

| Exact channel name | Electrode region / meaning | Current availability |
|---|---|---|
| `BTEleft SD` | Pair behind the **left ear**, near the mastoid; local same-side derivation | 68 recordings |
| `BTEright SD` | Pair behind the **right ear**, near the mastoid; local same-side derivation | 37 recordings |
| `CROSStop SD` | Cross-head derivation using electrodes behind **opposite ears** | All 105 recordings |

`CROSStop SD` does not mean a vertex/Cz electrode. This describes regions and derivations, not exact coordinates or polarity. The label alone does not establish a physical electrode count. Placement reference: [SeizeIT2 paper, Recording Setup and Figure 2](https://www.nature.com/articles/s41597-025-05580-x); local EEG JSON files specify `EEGPlacementScheme: behind-the-ear` and montage side.

### Channel combinations you can paste into the settings cell

Choose **one** option, replacing `EEG_CHANNELS` in the Python cell below. Also set `SUBJECTS` to a compatible group when choosing a side-specific channel.

| Experiment | `EEG_CHANNELS` value | Compatible current participants |
|---|---|---|
| Common cross-head channel | `["CROSStop SD"]` | All ten |
| Left local behind-ear only | `["BTEleft SD"]` | Left group below |
| Right local behind-ear only | `["BTEright SD"]` | Right group below |
| Left local + cross-head | `["BTEleft SD", "CROSStop SD"]` | Left group below |
| Right local + cross-head | `["BTEright SD", "CROSStop SD"]` | Right group below |
| Automatically use shared exact names | `None` | Intersection across your selected recordings; all ten currently share only `CROSStop SD` |

**Current left group:** `sub-001`, `sub-002`, `sub-004`, `sub-005`, `sub-006`, `sub-009`.

**Current right group:** `sub-003`, `sub-007`, `sub-008`, `sub-010`.

**Channel and participant settings must agree.** Changing `EEG_CHANNELS` does not automatically change `SUBJECTS`. For example, `EEG_CHANNELS=["BTEright SD"]` with `SUBJECTS=None` requests right-local EEG from every person, including left-only participants, so it must fail. Use the compatible right group above. The inventory now prints **ACTIVE SELECTION** before checking and names compatible participants in a channel error. Run the edited settings cell and rerun inventory; a failed inventory clears its old selection and prevents starting a session.

Example: a left-local-only experiment, without the cross-head channel:

```python
EEG_CHANNELS = ["BTEleft SD"]
SUBJECTS = ["sub-001", "sub-002", "sub-004", "sub-005", "sub-006", "sub-009"]
```

For a right-local-only experiment:

```python
EEG_CHANNELS = ["BTEright SD"]
SUBJECTS = ["sub-003", "sub-007", "sub-008", "sub-010"]
```

To include the cross-head signal, add `"CROSStop SD"` to that group's channel list. For the common-channel experiment, use `EEG_CHANNELS = ["CROSStop SD"]` and `SUBJECTS = None`.

**Both local ears together** (`["BTEleft SD", "BTEright SD"]`) and **all three channels** are unavailable within the current recordings. They can be selected in future recordings only if every selected EDF actually contains those names. Do not combine a left channel from one patient with a right channel from another patient to fabricate a two-channel example. Local left and right channels are not silently renamed or substituted.

These compatibility lists describe the current download; recheck inventory when adding participants. No forehead/central/other scalp channels are present to select here. An absent requested channel stops before waveform processing. Channel order is preserved, and the network automatically uses the selected input-channel count. Compatible channels do not guarantee enough eligible seizure/background examples: inspect patient readiness too.

### Deep learning remains the only model path

Every selection uses the same **three-block waveform CNN + stacked LSTM**. There is no shallow-model alternative or fallback. `CONV_WIDTHS`, `LSTM_HIDDEN_SIZE`, `LSTM_LAYERS`, `DROPOUT`, `MAX_EPOCHS`, `WINDOW_SEC` and `BATCH_SIZE` below let you tune architecture/context/training. Scikit-learn imports are for metrics only; PyTorch fits the model. The input remains filtered waveform samples, not a handcrafted feature-summary table.

After changing channels or participants, restart the kernel and run from the top to create a **new session**. Retrain for that channel selection; a previously saved model expects its saved channel order and input dimensions. The LSTM models time within each input window; it does not retain hidden state between windows.

The default input is 30 seconds at 128 Hz, filtered to 0.5–40 Hz. Increase `WINDOW_SEC` to show the model more past waveform, with increased RAM/computation. `STRIDE_SEC` controls spacing between decisions. `TARGET_FS=None` keeps native rates, which must match across recordings. Optional notch filtering removes a named narrow frequency; it does not summarize the signal. Filtering cannot guarantee removal of every artifact.

### Select which seizures define the binary target

| `SEIZURE_GROUP` | Positive target |
|---|---|
| `"all"` | Any annotated seizure (`sz` and its subtypes) |
| `"focal"` | `sz_foc` and every focal subtype, including focal-to-bilateral codes |
| `"focal_aware"` | `sz_foc_a` and its subtypes |
| `"focal_impaired_awareness"` | `sz_foc_ia` and its subtypes |
| `"generalized"` | `sz_gen` and its subtypes; absent from the current ten people |
| `"custom"` | Your `SEIZURE_PREFIXES` list, interpreted with `SEIZURE_MATCH` |

For focal forecasting, change **`SEIZURE_GROUP = "focal"`** in the settings cell. For exactly one annotation code:

```python
SEIZURE_GROUP = "custom"
SEIZURE_PREFIXES = ["sz_foc_ia_nm"]
SEIZURE_MATCH = "exact"
```

Use `"prefix"` to also include descendants such as `sz_foc_ia_nm_behavior`; prefix matching respects underscore boundaries. Multiple custom entries form a union. Presets always include their subtype descendants. These follow the [dataset's event-code definitions](https://raw.githubusercontent.com/OpenNeuroDatasets/ds005873/refs/heads/main/events.json).

The current 33 seizure annotations all have focal codes, so `"all"` and `"focal"` currently produce the same targets. The inventory prints actual codes, counts, patient availability and the selected rule before processing. An absent target gives a clear error. Code availability does not guarantee enough eligible examples after forecast/exclusion checks.

**This remains binary forecasting**, not multiclass subtype classification: `label=1` predicts the selected future event group; `label=0` is eligible background; `label=-1` is excluded. Nonselected seizures in the future target interval are excluded, not used as negative examples. Input/postictal/warning-gap exclusions cover all seizure types.

`STRATIFY_PATIENTS=True` balances selected eligible event types and class counts across whole patients for automatic 60/20/20 splitting. It never splits neighboring windows from the same patient across partitions. Exact subtype balance is not always possible. Rare subtypes/insufficient cases are reported; explicit `PATIENT_SPLITS` take priority. A new target requires a **new session and retraining**; saved sessions/models retain their original target rule.


### Ready-to-copy seizure-group configurations

**Default:** include every available seizure type in one binary positive group, with automatic patient stratification. This does not train a separate classifier per seizure type. `"all"` selects all seizure codes in the chosen EEG recordings; `SUBJECTS`, `RECORDINGS` and `EEG_CHANNELS` still control the data cohort.

Copy **one** option into the settings cell below, replacing its existing assignments. For every option keep:

```python
STRATIFY_PATIENTS = True
PATIENT_SPLITS = None
```

Supplying explicit `PATIENT_SPLITS` uses your assignments instead of the automatic stratifier. Patient separation remains enforced. Stratification balances selected classes/subtypes where feasible; it cannot create rare cases or guarantee every subtype in every partition.

**A. All seizures — shipped default**

```python
SESSION_NAME = "all_seizures"
SEIZURE_GROUP = "all"
```

**B. All focal seizures, including their subtypes**

```python
SESSION_NAME = "focal"
SEIZURE_GROUP = "focal"
```

**C. Focal-aware seizures**

```python
SESSION_NAME = "focal_aware"
SEIZURE_GROUP = "focal_aware"
```

**D. Focal impaired-awareness seizures**

```python
SESSION_NAME = "focal_impaired_awareness"
SEIZURE_GROUP = "focal_impaired_awareness"
```

**E. Generalized seizures — only when present in future data**

```python
SESSION_NAME = "generalized"
SEIZURE_GROUP = "generalized"
```

There are no generalized seizure codes in the current download, so option E currently stops with a no-matching-target message.

**F. Combine selected groups — here, focal-aware OR focal impaired-awareness**

```python
SESSION_NAME = "aware_or_impaired"
SEIZURE_GROUP = "custom"
SEIZURE_PREFIXES = ["sz_foc_a", "sz_foc_ia"]
SEIZURE_MATCH = "prefix"
```

**G. Exactly one annotation code**

```python
SESSION_NAME = "exact_focal_ia_nm"
SEIZURE_GROUP = "custom"
SEIZURE_PREFIXES = ["sz_foc_ia_nm"]
SEIZURE_MATCH = "exact"
```

For multiple exact codes, put them in the same list with `SEIZURE_MATCH="exact"`. Prefix mode includes subtype descendants; exact mode does not. `SEIZURE_PREFIXES` and `SEIZURE_MATCH` are used only by the `"custom"` preset.

`SESSION_NAME` labels an experiment; it does **not** change target labels. `SEIZURE_GROUP`/custom matching change the target. New sessions are named like `intermediate/<timestamp>_all_seizures_<unique-suffix>/` and still contain only `processed/`, `model/`, `report/`.

Restart the kernel and run from the top after choosing a configuration. Existing sessions retain their original settings. To inspect processing without fitting, additionally set `RUN_TRAINING=False` with any option. The model remains the waveform CNN + LSTM for every training option.


In [ ]:
PROJECT_DIR = None  # Set an absolute project folder only if the kernel starts elsewhere.
ROOT = (
    Path(PROJECT_DIR).expanduser().resolve()
    if PROJECT_DIR
    else next(
        (
            p
            for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "EEG_Seizure_Prediction.ipynb").is_file()
        ),
        Path.cwd(),
    ).resolve()
)
SOURCE_DIR = ROOT / "source"
OUTPUT_DIR = ROOT / "intermediate"
SESSION_NAME = "all_seizures"  # Session label; timestamp and unique suffix are automatic.
RUN_TRAINING = True  # False = save and inspect EEG waveforms only.
SUBJECTS = None  # Include all available participants.
# For CROSStop SD across all participants, use SUBJECTS = None.
# For BTEleft SD, use sub-001, sub-002, sub-004, sub-005, sub-006, sub-009.
RECORDINGS = None  # Optional list of exact full recording IDs.
EEG_CHANNELS = None

WINDOW_SEC = 30
STRIDE_SEC = 30
TARGET_FS = 128  # None = native (all selected rates must match).
BANDPASS_HZ = [0.5, 40.0]
NOTCH_HZ = None  # Example: 50.0, only when below both relevant Nyquist limits.
FILTER_WARMUP_SEC = 10
MAX_ABS_UV = None  # Optional amplitude rejection threshold; no default clipping.

MINIMUM_WARNING_MIN = 30
OCCURRENCE_PERIOD_MIN = 10
POSTICTAL_EXCLUSION_MIN = 30
SEIZURE_GROUP = "all"  # Default: one positive target covering every seizure type.
SEIZURE_PREFIXES = ["sz"]  # Used only for custom: one or more annotation codes.
SEIZURE_MATCH = (
    "prefix"  # Custom mode: "prefix" includes subtypes; "exact" matches codes only.
)
STRATIFY_PATIENTS = True  # Default: balance selected events/types across whole patients.
PATIENT_SPLITS = None  # Default: use automatic patient stratification (~60/20/20).
# Example for a planned four-patient experiment:
# PATIENT_SPLITS = {"train": ["sub-001", "sub-002"],
#                   "validation": ["sub-004"], "test": ["sub-007"]}
MIN_EVENTS_PER_PARTITION = 2

MAX_EPOCHS = 30
PATIENCE = 5
BATCH_SIZE = 32
LEARNING_RATE = 0.001
WEIGHT_DECAY = 0.0001
CONV_WIDTHS = [16, 32, 64]
LSTM_HIDDEN_SIZE = 64
LSTM_LAYERS = 2
DROPOUT = 0.2
DEVICE = "auto"  # "auto", "cpu", or "cuda".
CPU_THREADS = 2
SEED = 42

ALARM_THRESHOLDS = [0.1, 0.3, 0.5, 0.7, 0.9, 0.99, 1.0]
CONSECUTIVE_WINDOWS = 3
REFRACTORY_SEC = 2400
MAX_FALSE_ALARMS_PER_ELIGIBLE_DAY = 2

print("Source:", SOURCE_DIR)
print("Output:", OUTPUT_DIR)
print("Selected channels:", EEG_CHANNELS or "common exact names, determined below")

# Settings changed: require a successful new inventory before starting a session.
INVENTORY_READY = False


## 3. Session helpers and input checks — definitions

These functions validate settings and discover EEG files. No ECG/EMG/motion EDF is opened. Original files stay unchanged. Small JSON/CSV files record the settings and exclusions **inside the current session**, so each saved model remains understandable without a separate auditing system.

In [ ]:
def resolve_seizure_target(group, custom_codes, match):
    """Translate a readable preset into the same rule used by labeling and event scoring."""
    presets = {
        "all": ["sz"],
        "focal": ["sz_foc"],
        "focal_aware": ["sz_foc_a"],
        "focal_impaired_awareness": ["sz_foc_ia"],
        "generalized": ["sz_gen"],
    }
    if group not in set(presets) | {"custom"}:
        raise ValueError(
            f"Unknown SEIZURE_GROUP: {group}. Choose {list(presets) + ['custom']}."
        )
    patterns = custom_codes if group == "custom" else presets[group]
    mode = match if group == "custom" else "prefix"
    if mode not in ["prefix", "exact"]:
        raise ValueError("SEIZURE_MATCH must be prefix or exact.")
    if (
        not isinstance(patterns, list)
        or not patterns
        or any(
            not isinstance(x, str) or not re.fullmatch(r"sz(?:_[A-Za-z0-9]+)*", x)
            for x in patterns
        )
        or len(set(patterns)) != len(patterns)
    ):
        raise ValueError(
            "Select a nonempty list of unique seizure annotation codes, e.g. ['sz_foc_ia_nm']."
        )
    return {"seizure_group": group, "seizure_prefixes": list(patterns), "seizure_match": mode}


def is_target_seizure(code, labels):
    """Prefix matching respects underscore boundaries: aware does not match impaired-aware."""
    if not isinstance(code, str) or not code.startswith("sz"):
        return False
    patterns = labels["seizure_prefixes"]
    if labels.get("seizure_match", "prefix") == "exact":
        return code in patterns
    return any(code == pattern or code.startswith(pattern + "_") for pattern in patterns)


def seizure_annotation_summary(runs, labels):
    """Read EEG event TSVs only. Counts here are annotations, not forecast-eligible events."""
    frames = []
    for run in runs:
        events = pd.read_csv(run["events_file"], sep="\t")
        if "eventType" not in events:
            raise ValueError(f"Missing eventType in {run['events_file']}")
        seizures = events.loc[events.eventType.astype(str).str.startswith("sz"), ["eventType"]]
        frames.append(seizures.assign(subject=run["subject"]))
    combined = pd.concat(frames, ignore_index=True)
    if combined.empty:
        raise ValueError("No seizure annotations in the selected EEG recordings.")
    summary = (
        combined.groupby("eventType")
        .agg(events=("subject", "size"), patients=("subject", lambda x: sorted(set(x))))
        .reset_index()
    )
    summary["selected_target"] = summary.eventType.map(
        lambda code: is_target_seizure(code, labels)
    )
    return summary


def save_json(path, value):
    """Write readable metadata; undefined numerical metrics become JSON null."""

    def clean(item):
        if isinstance(item, dict):
            return {str(k): clean(v) for k, v in item.items()}
        if isinstance(item, (list, tuple)):
            return [clean(v) for v in item]
        if isinstance(item, np.generic):
            return clean(item.item())
        if isinstance(item, float) and not math.isfinite(item):
            return None
        if isinstance(item, Path):
            return str(item)
        return item

    Path(path).write_text(
        json.dumps(clean(value), indent=2, allow_nan=False), encoding="utf-8"
    )


def status(session, stage, error=None, **extra):
    """A failed step leaves its traceback beside the report, then raises normally."""
    path = session["report"] / "session.json"
    old = json.loads(path.read_text()) if path.exists() else {}
    old.update(stage=stage, updated_utc=datetime.now(timezone.utc).isoformat(), **extra)
    if error is not None:
        old["error"] = str(error)
        old["traceback"] = traceback.format_exc()
    save_json(path, old)


def validate_settings(c):
    raw, out = Path(c["source_dir"]).resolve(), Path(c["output_dir"]).resolve()
    if raw == out or raw in out.parents or out in raw.parents:
        raise ValueError(
            "Source and intermediate folders must be separate, non-nested folders."
        )
    if not raw.is_dir():
        raise FileNotFoundError(
            f"Source folder missing: {raw}. Put sub-* folders directly inside it."
        )
    if not re.fullmatch(r"[A-Za-z0-9_-]+", c["session_name"]):
        raise ValueError("SESSION_NAME needs only letters, numbers, underscores or hyphens.")
    for field in ["subjects", "recordings", "channels"]:
        value = c[field]
        if value is not None and (
            not isinstance(value, list)
            or not value
            or len(value) != len(set(value))
            or any(not isinstance(x, str) or not x for x in value)
        ):
            raise ValueError(f"{field} must be None or a nonempty list of unique names.")
    p, labels, model = c["preprocessing"], c["labels"], c["network"]
    for name in ["window_sec", "stride_sec"]:
        if not math.isfinite(p[name]) or p[name] <= 0:
            raise ValueError(f"{name} must be positive.")
    if p["filter_warmup_sec"] < 0 or not math.isfinite(p["filter_warmup_sec"]):
        raise ValueError("Invalid filter warm-up.")
    if p["target_fs"] is not None and (
        not math.isfinite(p["target_fs"]) or p["target_fs"] <= 0
    ):
        raise ValueError("Invalid target sampling rate.")
    if not (len(p["bandpass_hz"]) == 2 and 0 < p["bandpass_hz"][0] < p["bandpass_hz"][1]):
        raise ValueError("BANDPASS_HZ must have positive increasing lower/upper frequencies.")
    if p["max_abs_uv"] is not None and (
        not math.isfinite(p["max_abs_uv"]) or p["max_abs_uv"] <= 0
    ):
        raise ValueError("MAX_ABS_UV must be None or positive.")
    for key in ["minimum_warning_min", "postictal_exclusion_min"]:
        if not math.isfinite(labels[key]) or labels[key] < 0:
            raise ValueError(f"Invalid {key}.")
    if (
        not math.isfinite(labels["occurrence_period_min"])
        or labels["occurrence_period_min"] <= 0
    ):
        raise ValueError("Occurrence period must be positive.")
    if not labels["seizure_prefixes"] or any(
        not x.startswith("sz") for x in labels["seizure_prefixes"]
    ):
        raise ValueError("Select valid seizure prefixes, e.g. ['sz'].")
    if labels.get("seizure_match", "prefix") not in ["prefix", "exact"]:
        raise ValueError("Invalid seizure matching mode.")
    if not isinstance(c.get("stratify_patients", True), bool):
        raise ValueError("STRATIFY_PATIENTS must be True or False.")
    for key in [
        "max_epochs",
        "patience",
        "batch_size",
        "hidden_size",
        "lstm_layers",
        "cpu_threads",
    ]:
        if not isinstance(model[key], int) or model[key] < 1:
            raise ValueError(f"Invalid positive integer: {key}.")
    if len(model["conv_widths"]) != 3 or any(
        not isinstance(w, int) or w < 1 for w in model["conv_widths"]
    ):
        raise ValueError("CONV_WIDTHS needs three positive integer widths.")
    if (
        not 0 <= model["dropout"] < 1
        or model["learning_rate"] <= 0
        or model["weight_decay"] < 0
    ):
        raise ValueError("Invalid neural-network optimization settings.")
    if model["device"] not in ["auto", "cpu", "cuda"]:
        raise ValueError("DEVICE must be auto, cpu or cuda.")
    alarms = c["training"]["alarms"]
    if not alarms["thresholds"] or any(
        not math.isfinite(t) or not 0 <= t <= 1 for t in alarms["thresholds"]
    ):
        raise ValueError("Alarm thresholds must lie in [0,1].")
    if not isinstance(alarms["consecutive_windows"], int) or alarms["consecutive_windows"] < 1:
        raise ValueError("CONSECUTIVE_WINDOWS must be a positive integer.")
    if alarms["refractory_sec"] < 0 or alarms["max_false_alarms_per_day"] < 0:
        raise ValueError("Invalid alarm interval/budget.")
    if not isinstance(c["min_events"], int) or c["min_events"] < 1:
        raise ValueError("MIN_EVENTS_PER_PARTITION must be positive.")


def discover_eeg(c):
    """Match exact BIDS EEG IDs and validate metadata/channel availability before processing."""
    pattern = re.compile(
        r"(?P<id>(?P<subject>sub-[^_]+)_ses-[^_]+_task-[^_]+_run-[^_]+)_eeg\.edf$"
    )
    runs = []
    seen = set()
    for path in sorted(Path(c["source_dir"]).glob("sub-*/ses-*/eeg/*_eeg.edf")):
        match = pattern.fullmatch(path.name)
        if not match:
            continue
        identifier, subject = match["id"], match["subject"]
        if c["subjects"] is not None and subject not in c["subjects"]:
            continue
        if c["recordings"] is not None and identifier not in c["recordings"]:
            continue
        if identifier in seen:
            raise ValueError(f"Duplicate EEG recording ID: {identifier}")
        seen.add(identifier)
        sidecar = path.with_suffix(".json")
        events = path.with_name(identifier + "_events.tsv")
        if not sidecar.exists() or not events.exists():
            raise FileNotFoundError(f"Missing EEG JSON/events TSV for {identifier}")
        metadata = json.loads(sidecar.read_text(encoding="utf-8-sig"))
        with mne.io.read_raw_edf(path, preload=False, verbose=False) as raw:
            duration = raw.n_times / raw.info["sfreq"]
            if len(raw.annotations):
                raise ValueError(
                    f"{identifier}: EDF annotations need review for gaps/artifacts."
                )
            if abs(metadata.get("RecordingDuration", duration) - duration) > 1:
                raise ValueError(f"{identifier}: EDF/JSON duration mismatch.")
            if (
                abs(metadata.get("SamplingFrequency", raw.info["sfreq"]) - raw.info["sfreq"])
                > 1e-6
            ):
                raise ValueError(f"{identifier}: EDF/JSON sampling-rate mismatch.")
            if any(
                raw._orig_units.get(ch, "").lower() not in ["uv", "µv", "μv", "mv", "v"]
                for ch in raw.ch_names
            ):
                raise ValueError(
                    f"{identifier}: review non-voltage EEG units before using microvolt scaling."
                )
            runs.append(
                {
                    "id": identifier,
                    "subject": subject,
                    "file": str(path.resolve()),
                    "events_file": str(events.resolve()),
                    "sidecar_file": str(sidecar.resolve()),
                    "available_channels": list(raw.ch_names),
                    "native_fs": float(raw.info["sfreq"]),
                    "duration_sec": duration,
                }
            )
    if not runs:
        raise ValueError(
            "No selected EEG recordings. Check SOURCE_DIR and participant/recording names."
        )
    for field, observed in [("subjects", {r["subject"] for r in runs}), ("recordings", seen)]:
        if c[field] is not None and set(c[field]) - observed:
            raise ValueError(f"Selected {field} absent: {sorted(set(c[field]) - observed)}")
    common = set.intersection(*(set(r["available_channels"]) for r in runs))
    channels = c["channels"] if c["channels"] is not None else sorted(common)
    if not channels:
        raise ValueError(
            "No shared exact channel names. Select compatible participants and explicit channels."
        )
    incompatible = [r for r in runs if set(channels) - set(r["available_channels"])]
    if incompatible:
        bad_subjects = {r["subject"] for r in incompatible}
        compatible_subjects = sorted({r["subject"] for r in runs} - bad_subjects)
        first = incompatible[0]
        selection = "all available participants (SUBJECTS=None)" if c["subjects"] is None else repr(c["subjects"])
        suggestion = (f"For these channels, set SUBJECTS = {compatible_subjects!r}. "
                      if compatible_subjects else
                      "No currently selected participant has every requested channel in all selected recordings. Choose another channel combination or compatible recordings. ")
        raise ValueError(
            f"Channel/participant mismatch: requested {channels}; selected {selection}. "
            f"{first['id']} contains {first['available_channels']}. "
            f"{len(incompatible)} of {len(runs)} selected recordings are incompatible. "
            + suggestion + "Update the settings cell, run it, then rerun inventory. Channels must exist together within each recording; no left/right substitution is made."
        )
    for r in runs:
        target = c["preprocessing"]["target_fs"] or r["native_fs"]
        high = c["preprocessing"]["bandpass_hz"][1]
        if high >= min(target, r["native_fs"]) / 2:
            raise ValueError(
                f"{r['id']}: upper filter frequency must be below input and output Nyquist."
            )
        notch = c["preprocessing"]["notch_hz"]
        if notch is not None and (
            not math.isfinite(notch) or not 0 < notch < min(target, r["native_fs"]) / 2
        ):
            raise ValueError(
                "NOTCH_HZ must be positive and below input/output Nyquist, or None."
            )
    rates = {c["preprocessing"]["target_fs"] or r["native_fs"] for r in runs}
    if len(rates) != 1:
        raise ValueError("Mixed native sampling rates; choose a common TARGET_FS.")
    return runs, list(channels), float(next(iter(rates)))


def sample_count(seconds, rate):
    count = seconds * rate
    if not np.isclose(count, round(count), atol=1e-6, rtol=0):
        raise ValueError(f"{seconds}s does not fall on the {rate}Hz sample grid.")
    return int(round(count))


### Run: check the EEG inventory and chosen channels

**Output:** available channel groups, selected recording IDs and estimated processed-data size. This only reads EDF headers. The estimate excludes temporary conversion space, models and other existing sessions.

In [ ]:
# Invalidate previous successful results before attempting a new selection.
INVENTORY_READY = False
RUNS, SELECTED_CHANNELS, OUTPUT_FS = [], None, None
CONFIG = {
    "source_dir": str(Path(SOURCE_DIR).resolve()),
    "output_dir": str(Path(OUTPUT_DIR).resolve()),
    "session_name": SESSION_NAME,
    "subjects": SUBJECTS,
    "recordings": RECORDINGS,
    "channels": EEG_CHANNELS,
    "run_training": RUN_TRAINING,
    "splits": PATIENT_SPLITS,
    "stratify_patients": STRATIFY_PATIENTS,
    "min_events": MIN_EVENTS_PER_PARTITION,
    "seed": SEED,
    "preprocessing": {
        "window_sec": WINDOW_SEC,
        "stride_sec": STRIDE_SEC,
        "target_fs": TARGET_FS,
        "bandpass_hz": BANDPASS_HZ,
        "notch_hz": NOTCH_HZ,
        "filter_warmup_sec": FILTER_WARMUP_SEC,
        "max_abs_uv": MAX_ABS_UV,
    },
    "labels": {
        "minimum_warning_min": MINIMUM_WARNING_MIN,
        "occurrence_period_min": OCCURRENCE_PERIOD_MIN,
        "postictal_exclusion_min": POSTICTAL_EXCLUSION_MIN,
        "background_separation_min": 0,
        **resolve_seizure_target(SEIZURE_GROUP, SEIZURE_PREFIXES, SEIZURE_MATCH),
        "excluded_event_types": ["impd"],
    },
    "network": {
        "conv_widths": CONV_WIDTHS,
        "hidden_size": LSTM_HIDDEN_SIZE,
        "lstm_layers": LSTM_LAYERS,
        "dropout": DROPOUT,
        "max_epochs": MAX_EPOCHS,
        "patience": PATIENCE,
        "batch_size": BATCH_SIZE,
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
        "device": DEVICE,
        "cpu_threads": CPU_THREADS,
    },
    "training": {
        "alarms": {
            "thresholds": ALARM_THRESHOLDS,
            "consecutive_windows": CONSECUTIVE_WINDOWS,
            "refractory_sec": REFRACTORY_SEC,
            "max_false_alarms_per_day": MAX_FALSE_ALARMS_PER_ELIGIBLE_DAY,
        }
    },
}
print("ACTIVE SELECTION:", {name: CONFIG[name] for name in ["subjects", "recordings", "channels"]})
validate_settings(CONFIG)
RUNS, SELECTED_CHANNELS, OUTPUT_FS = discover_eeg(CONFIG)
inventory = pd.DataFrame(RUNS)
display(
    inventory.groupby(inventory.available_channels.map(tuple)).agg(
        recordings=("id", "size"), participants=("subject", lambda x: sorted(set(x)))
    )
)
display(inventory[["subject", "id", "available_channels", "duration_sec"]])
print("Model channel order:", SELECTED_CHANNELS)
print("One model input:", (len(SELECTED_CHANNELS), sample_count(WINDOW_SEC, OUTPUT_FS)))
estimated_bytes = (
    sum(max(0, int(r["duration_sec"] / STRIDE_SEC)) for r in RUNS)
    * len(SELECTED_CHANNELS)
    * sample_count(WINDOW_SEC, OUTPUT_FS)
    * 4
)
print(
    f"Conservative waveform estimate: {estimated_bytes / 1024**3:.2f} GiB per session before exclusions"
)


TYPE_INVENTORY = seizure_annotation_summary(RUNS, CONFIG["labels"])
display(TYPE_INVENTORY)
print(
    "Target selection:",
    {k: CONFIG["labels"][k] for k in ["seizure_group", "seizure_prefixes", "seizure_match"]},
)
print("Counts above are annotations BEFORE window coverage/artifact exclusions.")
if not TYPE_INVENTORY.selected_target.any():
    raise ValueError(
        "No annotations match the selected seizure target. Inspect the available codes above; choose another target or data cohort."
    )
selected_people = set().union(*TYPE_INVENTORY.loc[TYPE_INVENTORY.selected_target, "patients"])
if len(selected_people) < 3:
    print(
        "Target warning: annotations occur in fewer than three selected patients. Preparation is possible, but separate train/validation/test partitions with both classes will fail readiness."
    )

INVENTORY_READY = True
print("INVENTORY READY — the next cell can start a new session.")


### Run: start a new session

**Output:** one new timestamped folder, with `processed/`, `model/`, `report/`. Run this cell once per experiment. Rerunning it intentionally creates another session. Existing sessions are preserved. If changing settings, restart and run from the top rather than mixing variables from different experiments.

In [ ]:
if not globals().get("INVENTORY_READY", False):
    raise RuntimeError("Inventory has not completed for the current settings. Run the settings cell and finish the EEG inventory successfully before starting a session.")

session_id = (
    datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
    + "_"
    + SESSION_NAME
    + "_"
    + uuid4().hex[:6]
)
SESSION = {
    "root": Path(CONFIG["output_dir"]) / session_id,
    "config": copy.deepcopy(CONFIG),
    "channels": list(SELECTED_CHANNELS),
    "output_fs": OUTPUT_FS,
}
SESSION["root"].mkdir(parents=True, exist_ok=False)
for name in ["processed", "model", "report"]:
    SESSION[name] = SESSION["root"] / name
    SESSION[name].mkdir()
save_json(
    SESSION["report"] / "session.json",
    {
        "session": session_id,
        "stage": "created",
        "config": SESSION["config"],
        "channels": SESSION["channels"],
        "output_fs": OUTPUT_FS,
        "versions": {
            name: importlib.metadata.version(name)
            for name in ["numpy", "pandas", "scipy", "mne", "torch", "scikit-learn"]
        },
        "timing": "EEG EDF origin is the event TSV origin; whole-run annotation coverage is assumed",
    },
)
test_evaluated = False
print("NEW SESSION:", SESSION["root"])

TYPE_INVENTORY.to_csv(SESSION["report"] / "seizure_types.csv", index=False)


## 4. Forecast labels — definitions

At the end of the observed input, `t`, the target is a seizure onset in `[t + warning, t + warning + occurrence_period)`. Labels are **1 = positive**, **0 = eligible background**, **-1 = excluded**. The full future interval must be observed, even for positive examples. Seizure/postictal inputs, warning-gap seizures and impedance periods are excluded. Future annotations create targets; future waveform samples never enter model inputs.

Whole-run annotation coverage is an explicit assumption. An event-free recording still needs a valid annotation file; missing annotations are not treated as proof of background.

The same selector drives window labels and event/alarm scoring. `target_events` and `target_seizure_types` in the window CSV record selected event IDs/types as target metadata only. Every unique selected event is counted once for splitting, even when many windows target it or one window targets several events.

In [ ]:
# Function definitions: the algorithms are visible here in the notebook.
def read_events(path, duration):
    if path is None:
        raise ValueError("Missing event annotations; cannot assume background")
    events = pd.read_csv(path, sep="\t")
    if events.empty or not {"onset", "duration", "eventType"}.issubset(events.columns):
        raise ValueError("Invalid/empty event annotations")
    for _, e in events.iterrows():
        if (
            not np.isfinite(e.onset)
            or not np.isfinite(e.duration)
            or e.onset < 0
            or (e.duration <= 0)
            or (e.onset + e.duration > duration + 0.1)
        ):
            raise ValueError("Event outside recording or invalid duration")
        if not isinstance(e.eventType, str) or not (
            e.eventType.startswith("sz") or e.eventType in ("bckg", "impd")
        ):
            raise ValueError(f"Unknown event type: {e.eventType}")
    return events


def label_windows(starts, ends, events, duration, cfg):
    """Input ends are forecast times. Future intervals are half-open; exclusions win."""
    y = np.full(len(ends), -1, dtype=np.int8)
    reasons = np.full(len(ends), "unobserved_future", dtype=object)
    matched = np.full(len(ends), "", dtype=object)
    h, p = (cfg["minimum_warning_min"] * 60, cfg["occurrence_period_min"] * 60)
    post, sep = (cfg["postictal_exclusion_min"] * 60, cfg["background_separation_min"] * 60)
    for i, (a, b) in enumerate(zip(starts, ends)):
        if b + h + p > duration + 1e-08:
            continue
        y[i], reasons[i] = (0, "background")
        for idx, e in events.iterrows():
            selected = is_target_seizure(e.eventType, cfg)
            if selected and b + h <= e.onset < b + h + p:
                y[i], reasons[i], matched[i] = (1, "preictal", str(idx))
        for _, e in events.iterrows():
            seizure = e.eventType.startswith("sz")
            if seizure and a < e.onset + e.duration + post and (b > e.onset):
                y[i], reasons[i] = (-1, "ictal_or_postictal_input")
            elif seizure and b <= e.onset < b + h:
                y[i], reasons[i] = (-1, "seizure_inside_warning_gap")
            elif (
                seizure
                and not is_target_seizure(e.eventType, cfg)
                and (b + h <= e.onset < b + h + p)
            ):
                y[i], reasons[i] = (-1, "unselected_seizure")
            elif (
                seizure
                and y[i] == 0
                and (a < e.onset + e.duration + sep)
                and (b > e.onset - sep)
            ):
                y[i], reasons[i] = (-1, "background_separation")
            elif (
                e.eventType in cfg["excluded_event_types"]
                and a < e.onset + e.duration
                and (b > e.onset)
            ):
                y[i], reasons[i] = (-1, "excluded_event_input")
        if y[i] < 0:
            matched[i] = ""
    return (y, reasons, matched)


def annotate_targets(windows, events, labels):
    """Keep all selected event IDs/types per positive window as TARGET metadata, never predictors."""
    selected = events.loc[events.eventType.map(lambda code: is_target_seizure(code, labels))]
    horizon = labels["minimum_warning_min"] * 60
    period = labels["occurrence_period_min"] * 60
    all_targets, all_types = [], []
    for row in windows.itertuples():
        hits = (
            selected.loc[
                (selected.onset >= row.prediction_time_sec + horizon)
                & (selected.onset < row.prediction_time_sec + horizon + period)
            ]
            if row.label == 1
            else selected.iloc[:0]
        )
        targets = [
            {"row": str(idx), "type": str(event.eventType)} for idx, event in hits.iterrows()
        ]
        all_targets.append(json.dumps(targets, separators=(",", ":")))
        all_types.append(json.dumps(sorted({item["type"] for item in targets})))
    windows = windows.copy()
    windows["target_events"] = all_targets
    windows["target_seizure_types"] = all_types
    return windows


## 5. Clean EEG while retaining waveforms — definitions

For each eligible input window, read its waveform plus preceding filter warm-up. Apply a **causal Butterworth band-pass**, optional notch, then optional anti-alias downsampling. Only samples at/before the decision time are read. All channels retain their sample-by-sample time structure. No FFT-summary features, PCA or handcrafted averages enter the network.

The saved arrays are float32 **microvolts**, shaped **eligible examples × channels × samples**. Nonfinite/flat signals and optional excessive amplitudes are rejected. This is basic quality screening, not complete artifact removal. The temporary binary writer bounds RAM and is deleted after each `.npy` is finalized. Only retained windows occupy the final arrays; exclusion rows stay in one CSV for explanation.

In [ ]:
def filter_window(data_volts, fs, cfg):
    """Input: channels × observed samples, including past warm-up. Output: EEG microvolts."""
    p = cfg["preprocessing"]
    sos = butter(4, p["bandpass_hz"], btype="bandpass", fs=fs, output="sos")
    filtered = sosfilt(sos, data_volts, axis=-1)
    if p["notch_hz"] is not None:
        b, a = iirnotch(p["notch_hz"], Q=30, fs=fs)
        filtered = sosfilt(tf2sos(b, a), filtered, axis=-1)
    target = p["target_fs"] or fs
    ratio = Fraction(target / fs).limit_denominator(10000)
    if not np.isclose(float(ratio), target / fs, atol=1e-10, rtol=0):
        raise ValueError("Sampling-rate ratio cannot be represented accurately.")
    if ratio.numerator != ratio.denominator:
        filtered = resample_poly(filtered, ratio.numerator, ratio.denominator, axis=-1)
    expected = sample_count(p["window_sec"], target)
    warm = sample_count(p["filter_warmup_sec"], target)
    if filtered.shape[-1] != expected + warm:
        raise ValueError("Resampled sample count differs from expected window/warm-up.")
    return np.asarray(filtered[:, warm:] * 1e6, dtype=np.float32)


def prepare_eeg(session, runs):
    """Stream waveforms to disk. Return one small metadata table, not a dense feature matrix."""
    if any(session["processed"].iterdir()):
        raise RuntimeError(
            "This session already has processed files. Start a new session before processing again."
        )
    cfg, frames, records = session["config"], [], []
    p = cfg["preprocessing"]
    started = time.perf_counter()
    status(session, "processing")
    try:
        for number, run in enumerate(runs):
            print(f"[{number+1}/{len(runs)}] {run['id']}", flush=True)
            with mne.io.read_raw_edf(run["file"], preload=False, verbose=False) as raw:
                raw.pick(session["channels"])
                fs = raw.info["sfreq"]
                # Every observed input ends here. Separate runs are never concatenated.
                first_end = p["filter_warmup_sec"] + p["window_sec"]
                ends = np.arange(first_end, run["duration_sec"] + 1e-7, p["stride_sec"])
                starts = ends - p["window_sec"]
                support = starts - p["filter_warmup_sec"]
                events = read_events(run["events_file"], run["duration_sec"])
                if "recordingDuration" in events and not np.allclose(
                    events.recordingDuration, run["duration_sec"], atol=1, rtol=0
                ):
                    raise ValueError(f"{run['id']}: event TSV duration differs from EEG.")
                labels, reasons, event_rows = label_windows(
                    support, ends, events, run["duration_sec"], cfg["labels"]
                )
                count = int((labels >= 0).sum())
                samples = sample_count(p["window_sec"], session["output_fs"])
                estimate = count * len(session["channels"]) * samples * 4
                if shutil.disk_usage(session["processed"]).free < estimate * 2 + 100 * 1024**2:
                    raise OSError(
                        "Not enough free disk space for the recording and temporary conversion."
                    )
                stem = f"eeg_{number+1:04d}"
                temporary = session["processed"] / (stem + ".bin")
                sample_indices = np.full(len(ends), -1, dtype=int)
                kept = 0
                with temporary.open("wb") as stream:
                    for i in np.flatnonzero(labels >= 0):
                        left, right = sample_count(support[i], fs), sample_count(ends[i], fs)
                        observed = raw.get_data(start=left, stop=right)
                        reason = None
                        if not np.isfinite(observed).all():
                            reason = "nonfinite_signal"
                        elif np.any(np.ptp(observed, axis=1) == 0):
                            reason = "flat_channel"
                        elif (
                            p["max_abs_uv"] is not None
                            and np.max(np.abs(observed)) * 1e6 > p["max_abs_uv"]
                        ):
                            reason = "amplitude_limit"
                        if reason is None:
                            waveform = filter_window(observed, fs, cfg)
                            if not np.isfinite(waveform).all():
                                reason = "nonfinite_processed_signal"
                        if reason is not None:
                            labels[i], reasons[i], event_rows[i] = -1, reason, ""
                            continue
                        waveform.tofile(stream)
                        sample_indices[i] = kept
                        kept += 1
                shape = (kept, len(session["channels"]), samples)
                filename = stem + ".npy" if kept else None
                if kept:
                    source = np.memmap(temporary, mode="r", dtype="float32", shape=shape)
                    destination = np.lib.format.open_memmap(
                        session["processed"] / filename,
                        mode="w+",
                        dtype="float32",
                        shape=shape,
                    )
                    for begin in range(0, kept, 32):
                        destination[begin : begin + 32] = source[begin : begin + 32]
                    destination.flush()
                    destination._mmap.close()
                    source._mmap.close()
                temporary.unlink()
                frames.append(
                    pd.DataFrame(
                        {
                            "subject": run["subject"],
                            "recording_id": run["id"],
                            "array_file": filename,
                            "sample_index": sample_indices,
                            "input_start_sec": starts,
                            "processing_support_start_sec": support,
                            "prediction_time_sec": ends,
                            "label": labels,
                            "eligible": labels >= 0,
                            "reason": reasons,
                            "event_row": event_rows,
                        }
                    )
                )
                frames[-1] = annotate_targets(frames[-1], events, cfg["labels"])
                fingerprints = {
                    name: {
                        "bytes": Path(run[name]).stat().st_size,
                        "mtime_ns": Path(run[name]).stat().st_mtime_ns,
                    }
                    for name in ["file", "events_file", "sidecar_file"]
                }
                records.append(
                    {
                        **run,
                        "array_file": filename,
                        "shape": list(shape),
                        "channels": session["channels"],
                        "units": "microvolts",
                        "source_fingerprints": fingerprints,
                    }
                )
                print(
                    f"  retained {kept:,}/{len(ends):,} windows | positives {int((labels==1).sum()):,}",
                    flush=True,
                )
        windows = pd.concat(frames, ignore_index=True)
        windows.to_csv(session["processed"] / "windows.csv", index=False)
        save_json(session["processed"] / "recordings.json", records)
        if not windows.eligible.any():
            raise ValueError(
                "No eligible windows. Inspect exclusions, forecast horizon and annotation coverage."
            )
        status(
            session,
            "prepared",
            total_windows=len(windows),
            eligible_windows=int(windows.eligible.sum()),
            positive_windows=int((windows.label == 1).sum()),
            processing_seconds=time.perf_counter() - started,
        )
        return windows, records
    except Exception as error:
        status(session, "processing_failed", error)
        raise


### Run: process EEG and inspect the saved waveforms

**Output:** per-recording progress, patient/class counts, excluded-window reasons, and raw/filtered waveform plots. The first processed channel is previewed; this does not inspect every artifact. Arrays remain on disk after closing Jupyter.

In [ ]:
WINDOWS, RECORDS = prepare_eeg(SESSION, RUNS)
ELIGIBLE = WINDOWS.loc[WINDOWS.eligible].reset_index(drop=True)
display(WINDOWS.groupby(["subject", "reason"]).size().unstack(fill_value=0))
display(ELIGIBLE.groupby("subject").label.agg(windows="size", positive="sum"))
example = ELIGIBLE.iloc[0]
record = next(r for r in RECORDS if r["id"] == example.recording_id)
array = np.load(SESSION["processed"] / example.array_file, mmap_mode="r")
waveform = array[int(example.sample_index)].copy()
array._mmap.close()
with mne.io.read_raw_edf(record["file"], preload=False, verbose=False) as raw:
    raw.pick(SESSION["channels"])
    original = (
        raw.get_data(
            start=sample_count(example.input_start_sec, raw.info["sfreq"]),
            stop=sample_count(example.prediction_time_sec, raw.info["sfreq"]),
        )[0]
        * 1e6
    )
    original_fs = raw.info["sfreq"]
fig, axes = plt.subplots(1, 2, figsize=(12, 3))
axes[0].plot(np.arange(len(original)) / original_fs, original, alpha=0.5, label="original")
axes[0].plot(
    np.arange(waveform.shape[-1]) / SESSION["output_fs"], waveform[0], label="filtered"
)
axes[0].set(
    xlabel="Seconds inside observed input", ylabel="microvolts", title=SESSION["channels"][0]
)
axes[0].legend()
for values, rate, name in [
    (original, original_fs, "original"),
    (waveform[0], SESSION["output_fs"], "filtered"),
]:
    frequency, power = welch(values, fs=rate, nperseg=min(len(values), int(rate * 2)))
    axes[1].semilogy(frequency, np.maximum(power, 1e-15), label=name)
axes[1].set(
    xlabel="Hz",
    ylabel="Power / Hz",
    xlim=(0, min(original_fs, SESSION["output_fs"]) / 2),
    title="Spectrum preview only; not model features",
)
axes[1].legend()
plt.tight_layout()
plt.show()
print("PREPARATION COMPLETE:", SESSION["processed"])
print("Saved model input shape per example:", waveform.shape)


## 6. Patient separation and training-only normalization — definitions

The default uses a seeded patient-level stratification search with roughly 60% training / 20% validation / 20% test. It balances eligible selected-subtype event counts and positive/background windows, while keeping each person intact. Small cohorts are searched exhaustively at those sizes; larger cohorts use a bounded seeded search. The selection uses annotation/class counts, never model scores. A valid allocation is not guaranteed for a sparse target. Set `STRATIFY_PATIENTS=False` to recover sorted draft splitting. Use explicit `PATIENT_SPLITS` for a planned study. All windows from a person stay together. Each partition needs both labels and enough distinct eligible seizures; positive-window counts alone are insufficient. Three distinct eligible people are the structural minimum, not evidence of a reliable study. Two people can prepare data with `RUN_TRAINING=False`.

Normalization learns **one mean and standard deviation per channel from training waveforms only**. It does not reduce the waveform to those numbers: every normalized sample still enters the network. Global channel normalization preserves between-window amplitude differences, unlike normalizing each window independently. DataLoader reads mini-batches from memory-mapped files; the full waveform dataset is not loaded into RAM. Windows and training metadata remain small in-memory tables.

Review the printed per-partition subtype counts and `report/patient_splits.json` before interpreting scores. Some types cannot appear in every partition when they occur in fewer than three people. Narrower target selections can make a previously valid study impossible; add eligible patients or explicitly redesign the study, without a random-window fallback.

In [ ]:
def eligible_seizure_table(meta):
    """Deduplicate (patient, recording, original event row); windows are not independent seizures."""
    entries = {}
    if "target_events" not in meta:
        raise ValueError(
            "Target metadata missing. Start a new session and regenerate labels for this selection."
        )
    for row in meta.loc[meta.label == 1].itertuples():
        for target in json.loads(row.target_events):
            key = (row.subject, row.recording_id, str(target["row"]))
            item = {
                "subject": row.subject,
                "recording_id": row.recording_id,
                "event_row": str(target["row"]),
                "seizure_type": str(target["type"]),
            }
            if key in entries and entries[key] != item:
                raise ValueError("Conflicting target annotations for the same original event.")
            entries[key] = item
    return pd.DataFrame(
        list(entries.values()),
        columns=["subject", "recording_id", "event_row", "seizure_type"],
    )


def stratified_patient_split(meta, cfg):
    """Balance class windows and selected subtype event counts while assigning WHOLE people."""
    people = sorted(meta.subject.unique())
    n = len(people)
    if n < 3:
        raise ValueError(
            "Need at least three eligible people for separate train/validation/test."
        )
    events = eligible_seizure_table(meta)
    if events.subject.nunique() < 3:
        raise ValueError(
            "Selected target has eligible seizures in fewer than three patients. Add eligible patients or use preparation-only mode; whole-patient stratification cannot create missing cases."
        )
    class_counts = pd.crosstab(meta.subject, meta.label).reindex(
        index=people, columns=[0, 1], fill_value=0
    )
    type_counts = pd.crosstab(events.subject, events.seizure_type).reindex(
        index=people, fill_value=0
    )
    values = np.column_stack([class_counts.to_numpy(), type_counts.to_numpy()]).astype(float)
    totals = values.sum(axis=0)
    n_train = max(1, min(int(n * 0.6), n - 2))
    n_val = max(1, min(int(n * 0.2), n - n_train - 1))
    fractions = np.array([n_train, n_val, n - n_train - n_val], dtype=float) / n
    rng = np.random.default_rng(cfg["seed"])
    order = rng.permutation(n)
    possible = math.comb(n, n_train) * math.comb(n - n_train, n_val)

    def candidates():
        if possible <= 5000:
            # Exhaust small cohorts instead of trusting a few random allocations.
            for train in itertools.combinations(order.tolist(), n_train):
                remaining = [i for i in order if i not in train]
                for validation in itertools.combinations(remaining, n_val):
                    test = [i for i in remaining if i not in validation]
                    yield [list(train), list(validation), test]
        else:
            yield [
                list(range(n_train)),
                list(range(n_train, n_train + n_val)),
                list(range(n_train + n_val, n)),
            ]
            for _ in range(5000):
                perm = rng.permutation(n)
                yield [
                    perm[:n_train],
                    perm[n_train : n_train + n_val],
                    perm[n_train + n_val :],
                ]

    best, best_score = None, np.inf
    widespread = (type_counts.to_numpy() > 0).sum(axis=0) >= 3
    for groups in candidates():
        summed = np.vstack([values[group].sum(axis=0) for group in groups])
        # Every partition must contain background, positive windows and enough unique selected events.
        if (summed[:, :2] <= 0).any() or (summed[:, 2:].sum(axis=1) < cfg["min_events"]).any():
            continue
        shares = summed / np.maximum(totals, 1)
        missing = ((summed[:, 2:] == 0) & widespread[None, :]).sum()
        score = float(np.mean((shares - fractions[:, None]) ** 2) + missing)
        if score < best_score:
            best, best_score = groups, score
    if best is None:
        suffix = (
            "after checking all allocations at the default group sizes"
            if possible <= 5000
            else "within the seeded search budget"
        )
        raise ValueError(
            f"No valid whole-patient split found {suffix}. Need both classes and {cfg['min_events']} selected events per partition. Add data or supply explicit disjoint PATIENT_SPLITS; no random-window fallback is used."
        )
    return {
        name: sorted(people[int(i)] for i in group)
        for name, group in zip(["train", "validation", "test"], best)
    }


def patient_partitions(meta, cfg):
    people = sorted(meta.subject.unique())
    if len(people) < 3:
        raise ValueError(
            "Need at least three eligible people for separate train/validation/test. Use preparation-only mode for two."
        )
    splits = cfg["splits"]
    if splits is None and cfg.get("stratify_patients", True):
        splits = stratified_patient_split(meta, cfg)
    elif splits is None:
        n_train = max(1, min(int(len(people) * 0.6), len(people) - 2))
        n_val = max(1, min(int(len(people) * 0.2), len(people) - n_train - 1))
        splits = {
            "train": people[:n_train],
            "validation": people[n_train : n_train + n_val],
            "test": people[n_train + n_val :],
        }
    if set(splits) != {"train", "validation", "test"}:
        raise ValueError("PATIENT_SPLITS needs train, validation and test lists.")
    if any(not isinstance(v, list) or not v or len(v) != len(set(v)) for v in splits.values()):
        raise ValueError("Each partition needs a nonempty list of unique patient IDs.")
    flattened = [person for group in splits.values() for person in group]
    if len(flattened) != len(set(flattened)):
        raise ValueError(
            "A patient appears in multiple partitions; no patient overlap is allowed."
        )
    if set(flattened) != set(people):
        raise ValueError(
            "Splits must assign every eligible selected patient exactly once. Adjust SUBJECTS to exclude people deliberately."
        )
    ledger = eligible_seizure_table(meta)
    partitions, counts = {}, []
    for name, people_in_group in splits.items():
        rows = meta.loc[meta.subject.isin(people_in_group)].reset_index(drop=True)
        events = ledger.loc[ledger.subject.isin(people_in_group)]
        counts.append(
            {
                "partition": name,
                "patients": people_in_group,
                "windows": len(rows),
                "positive_windows": int(rows.label.sum()),
                "eligible_seizures": len(events),
                "prevalence": float(rows.label.mean()),
                "seizure_type_counts": events.seizure_type.value_counts().to_dict(),
            }
        )
        if set(rows.label) != {0, 1} or len(events) < cfg["min_events"]:
            raise ValueError(
                f"{name}: needs both classes and {cfg['min_events']} eligible selected seizures; found {counts[-1]}"
            )
        partitions[name] = rows
    coverage = ledger.groupby("seizure_type").subject.nunique()
    rare = coverage[coverage < 3].to_dict()
    if rare:
        print(
            "Subtype coverage warning: these types occur in fewer than three eligible patients and cannot appear in every partition:",
            rare,
        )
    return partitions, counts


def training_normalization(folder, metadata):
    """Streaming channel statistics; validation/test samples are never consulted."""
    total, sums, squares = 0, None, None
    for filename, rows in metadata.groupby("array_file", sort=False):
        array = np.load(Path(folder) / filename, mmap_mode="r")
        try:
            for start in range(0, len(rows), 32):
                ids = rows.iloc[start : start + 32].sample_index.to_numpy(dtype=int)
                values = np.asarray(array[ids], dtype=np.float64)
                block_sum = values.sum(axis=(0, 2))
                block_squares = np.square(values).sum(axis=(0, 2))
                sums = block_sum if sums is None else sums + block_sum
                squares = block_squares if squares is None else squares + block_squares
                total += values.shape[0] * values.shape[2]
        finally:
            array._mmap.close()
    mean = sums / total
    std = np.sqrt(np.maximum(squares / total - mean**2, 1e-12))
    return {
        "mean_uv": mean.tolist(),
        "std_uv": std.tolist(),
        "training_samples_per_channel": total,
    }


class WaveformDataset(Dataset):
    """Return complete channel × time windows, opening only disk-backed arrays."""

    def __init__(self, folder, rows, normalization):
        self.folder = Path(folder)
        self.rows = rows.reset_index(drop=True)
        self.mean = np.asarray(normalization["mean_uv"], dtype=np.float32)[:, None]
        self.std = np.asarray(normalization["std_uv"], dtype=np.float32)[:, None]
        self.arrays = {}

    def __len__(self):
        return len(self.rows)

    def __getitem__(self, index):
        row = self.rows.iloc[index]
        if row.array_file not in self.arrays:
            self.arrays[row.array_file] = np.load(self.folder / row.array_file, mmap_mode="r")
        # Copy this window only, so tensors do not hold mutable/read-only mmap views.
        values = np.array(
            self.arrays[row.array_file][int(row.sample_index)], dtype=np.float32, copy=True
        )
        values = (values - self.mean) / self.std
        return torch.from_numpy(values), torch.tensor(float(row.label), dtype=torch.float32)

    def close(self):
        for array in self.arrays.values():
            array._mmap.close()
        self.arrays.clear()


## 7. Deep waveform model — definitions

Three convolutional blocks learn time-local patterns, reducing sequence length inside the model. A stacked, unidirectional LSTM then reads the ordered learned sequence and produces one risk score. This is learned representation building from waveforms, rather than a fixed table of handcrafted summaries. Each input window is independent; hidden state resets between examples.

Training uses AdamW and class-weighted binary cross-entropy. The positive weight comes from training labels only. Early stopping chooses the best validation average precision; the test set is untouched. Using the same validation people for early stopping and threshold selection makes validation scores optimistic; the final test remains separate.

[PyTorch LSTM documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.LSTM.html) · [SciPy causal SOS filtering](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.sosfilt.html)

In [ ]:
class EEGConvLSTM(nn.Module):
    """Input: batch × channels × samples. Output: one unbounded logit per example."""

    def __init__(self, n_channels, widths, hidden_size, lstm_layers, dropout):
        super().__init__()
        blocks = []
        incoming = n_channels
        for width in widths:
            blocks.extend(
                [
                    nn.Conv1d(incoming, width, kernel_size=9, stride=4, padding=4),
                    nn.GroupNorm(1, width),
                    nn.GELU(),
                    nn.Dropout(dropout),
                ]
            )
            incoming = width
        self.convolutions = nn.Sequential(*blocks)
        self.lstm = nn.LSTM(
            input_size=incoming,
            hidden_size=hidden_size,
            num_layers=lstm_layers,
            batch_first=True,
            dropout=dropout if lstm_layers > 1 else 0.0,
        )
        self.output = nn.Sequential(nn.Dropout(dropout), nn.Linear(hidden_size, 1))

    def forward(self, waveform):
        # Convolution uses channels first; LSTM expects sequence steps before features.
        sequence = self.convolutions(waveform).transpose(1, 2)
        _, (hidden, _) = self.lstm(sequence)
        return self.output(hidden[-1]).squeeze(1)


def make_network(n_channels, cfg):
    settings = cfg["network"]
    return EEGConvLSTM(
        n_channels,
        settings["conv_widths"],
        settings["hidden_size"],
        settings["lstm_layers"],
        settings["dropout"],
    )


def predict_waveforms(model, loader, device):
    model.eval()
    probabilities = []
    with torch.inference_mode():
        for waveforms, _ in loader:
            values = torch.sigmoid(model(waveforms.to(device))).cpu().numpy()
            probabilities.append(values)
    return np.concatenate(probabilities).astype(float)


def train_network(session, train_rows, validation_rows, normalization):
    if (session["model"] / "model.pt").exists():
        raise RuntimeError(
            "This session already has model weights. Start a new session for another fit."
        )
    cfg, settings = session["config"], session["config"]["network"]
    random.seed(cfg["seed"])
    np.random.seed(cfg["seed"])
    torch.manual_seed(cfg["seed"])
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(cfg["seed"])
    torch.set_num_threads(settings["cpu_threads"])
    device = torch.device(
        "cuda"
        if settings["device"] == "auto" and torch.cuda.is_available()
        else "cpu" if settings["device"] == "auto" else settings["device"]
    )
    if device.type == "cuda" and not torch.cuda.is_available():
        raise ValueError("CUDA requested but unavailable; select cpu or auto.")
    model = make_network(len(session["channels"]), cfg).to(device)
    train_set = WaveformDataset(session["processed"], train_rows, normalization)
    validation_set = WaveformDataset(session["processed"], validation_rows, normalization)
    generator = torch.Generator().manual_seed(cfg["seed"])
    train_loader = DataLoader(
        train_set,
        batch_size=settings["batch_size"],
        shuffle=True,
        num_workers=0,
        generator=generator,
    )
    validation_loader = DataLoader(
        validation_set, batch_size=settings["batch_size"], shuffle=False, num_workers=0
    )
    positives = int(train_rows.label.sum())
    weight = (len(train_rows) - positives) / positives
    loss_function = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(weight, device=device))
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=settings["learning_rate"], weight_decay=settings["weight_decay"]
    )
    best_score, best_state, best_epoch, wait = -np.inf, None, None, 0
    history = []
    status(session, "training", device=str(device), training_positive_weight=weight)
    try:
        print(model)
        print(
            "Trainable parameters:",
            sum(p.numel() for p in model.parameters()),
            "| Device:",
            device,
        )
        for epoch in range(1, settings["max_epochs"] + 1):
            model.train()
            loss_sum = 0.0
            started = time.perf_counter()
            for waveforms, labels in train_loader:
                optimizer.zero_grad(set_to_none=True)
                loss = loss_function(model(waveforms.to(device)), labels.to(device))
                if not torch.isfinite(loss):
                    raise ValueError(
                        "Nonfinite loss: inspect waveform amplitudes and normalization."
                    )
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
                loss_sum += float(loss.item()) * len(labels)
            probabilities = predict_waveforms(model, validation_loader, device)
            score = float(average_precision_score(validation_rows.label, probabilities))
            history.append(
                {
                    "epoch": epoch,
                    "training_loss": loss_sum / len(train_rows),
                    "validation_average_precision": score,
                    "seconds": time.perf_counter() - started,
                }
            )
            pd.DataFrame(history).to_csv(session["report"] / "history.csv", index=False)
            print(
                f"Epoch {epoch}: loss {history[-1]['training_loss']:.4f} | validation AP {score:.4f} | {history[-1]['seconds']:.1f}s",
                flush=True,
            )
            if score > best_score:
                best_score, best_epoch, wait = score, epoch, 0
                best_state = {
                    key: value.detach().cpu().clone()
                    for key, value in model.state_dict().items()
                }
                # Save the best weights immediately; interruption does not lose completed epochs.
                torch.save(
                    {
                        "format": "eeg_conv_lstm_v1",
                        "state_dict": best_state,
                        "config": cfg,
                        "normalization": normalization,
                        "channels": session["channels"],
                        "output_fs": session["output_fs"],
                        "best_epoch": best_epoch,
                        "validation_average_precision": best_score,
                        "threshold": None,
                    },
                    session["model"] / "model.pt",
                )
            else:
                wait += 1
                if wait >= settings["patience"]:
                    print("Early stopping: validation AP stopped improving.")
                    break
        if best_state is None:
            raise RuntimeError("No valid training epoch completed.")
        model.load_state_dict(best_state)
        model.eval()
        status(
            session, "trained", best_epoch=best_epoch, validation_average_precision=best_score
        )
        return model, device, pd.DataFrame(history)
    except Exception as error:
        status(session, "training_failed", error)
        raise
    finally:
        train_set.close()
        validation_set.close()


### Run: check patients, normalize and train

**Output:** separate patient lists, class/event counts, the actual batch shape, network architecture and one progress line per epoch. Parameters update from training only. Best weights are saved in `model/model.pt`. A failed readiness check requires correcting the selected data or study design, not mixing windows across patients.

In [ ]:
if SESSION["config"]["run_training"]:
    PARTITIONS, COUNTS = patient_partitions(ELIGIBLE, SESSION["config"])
    display(pd.DataFrame(COUNTS))
    type_balance = (
        pd.DataFrame({row["partition"]: row["seizure_type_counts"] for row in COUNTS})
        .fillna(0)
        .astype(int)
    )
    display(type_balance)
    save_json(
        SESSION["report"] / "patient_splits.json",
        {
            "method": (
                "explicit"
                if SESSION["config"]["splits"] is not None
                else (
                    "patient_stratified"
                    if SESSION["config"].get("stratify_patients", True)
                    else "sorted"
                )
            ),
            "seed": SESSION["config"]["seed"],
            "counts": COUNTS,
            "note": "Assignments balance selected annotation events/eligible classes, never model scores. No patient appears in multiple partitions.",
        },
    )
    NORMALIZATION = training_normalization(SESSION["processed"], PARTITIONS["train"])
    probe = WaveformDataset(SESSION["processed"], PARTITIONS["train"], NORMALIZATION)
    try:
        example_batch, example_labels = next(
            iter(
                DataLoader(
                    probe, batch_size=SESSION["config"]["network"]["batch_size"], num_workers=0
                )
            )
        )
        print(
            "Actual input tensor:",
            tuple(example_batch.shape),
            "| labels:",
            tuple(example_labels.shape),
        )
        assert example_batch.ndim == 3 and torch.isfinite(example_batch).all()
    finally:
        probe.close()
    MODEL, MODEL_DEVICE, HISTORY = train_network(
        SESSION, PARTITIONS["train"], PARTITIONS["validation"], NORMALIZATION
    )
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].plot(HISTORY.epoch, HISTORY.training_loss)
    axes[0].set(xlabel="Epoch", ylabel="Weighted training loss")
    axes[1].plot(HISTORY.epoch, HISTORY.validation_average_precision)
    axes[1].set(xlabel="Epoch", ylabel="Validation average precision")
    plt.tight_layout()
    plt.savefig(SESSION["report"] / "learning_curve.png", dpi=140)
    plt.show()
else:
    print("Preparation-only mode: training skipped.")


## 8. Validation threshold and held-out report — definitions

Window scores and seizure-event warnings answer different questions. The validation patients choose the alarm threshold under the configured false-alarm budget. The test patients are evaluated after all choices are frozen. Consecutive decisions and a refractory interval suppress repeated alarms; gaps reset streaks. False alarms are normalized by **eligible decision support**, not by unobserved/excluded hours. The saved report states this limitation.

Class-weighted scores are not independently calibrated probabilities. Compare average precision with test prevalence, inspect seizure sensitivity and false alarms together, and report the number of eligible events. An all-quiet threshold can meet the false-alarm budget while warning no seizures.

In [ ]:
# Function definitions: the algorithms are visible here in the notebook.
def alarm_metrics(frame, probabilities, threshold, config, events_by_run):
    """Conditional evaluation on eligible windows only; no claim of continuous prospective FAR."""
    probabilities = np.asarray(probabilities, dtype=float)
    if (
        probabilities.shape != (len(frame),)
        or not np.isfinite(probabilities).all()
        or ((probabilities < 0) | (probabilities > 1)).any()
    ):
        raise ValueError("Alarm probabilities must match rows and lie in [0, 1]")
    if not np.isfinite(threshold) or not 0 <= threshold <= 1:
        raise ValueError("Invalid alarm threshold")
    data = frame.copy()
    data["probability"] = probabilities
    alarms = []
    eligible_events = set()
    warned = set()
    event_records = []
    step = config["preprocessing"]["stride_sec"]
    exposure_step = min(step, config["preprocessing"]["window_sec"])
    horizon = config["labels"]["minimum_warning_min"] * 60
    period = config["labels"]["occurrence_period_min"] * 60
    alarmcfg = config["training"]["alarms"]
    warning_cells = 0
    warning_seconds = 0.0
    exposure_seconds = 0.0
    for run, block in data.groupby("recording_id", sort=False):
        block = block.sort_values("prediction_time_sec")
        times = block.prediction_time_sec.to_numpy(dtype=float)
        if not np.isfinite(times).all() or (np.diff(times) <= 0).any():
            raise ValueError(
                "Prediction timestamps must be finite and unique within each recording"
            )
        if (np.diff(times) < step - 1e-06).any():
            raise ValueError("Prediction decisions closer together than configured stride")
        events = events_by_run[run]
        selected = events.loc[
            events.eventType.map(lambda code: is_target_seizure(code, config["labels"]))
        ]
        last = None
        last_alarm = -np.inf
        streak = 0
        warning_until = -np.inf
        for row in block.itertuples():
            t = row.prediction_time_sec
            hits = selected.index[
                (selected.onset >= t + horizon) & (selected.onset < t + horizon + period)
            ].tolist()
            eligible_events.update(((run, str(i)) for i in hits))
            if last is None or abs(t - last - step) > 1e-06:
                streak = 0
            streak = streak + 1 if row.probability >= threshold else 0
            if (
                streak >= alarmcfg["consecutive_windows"]
                and t - last_alarm >= alarmcfg["refractory_sec"]
            ):
                last_alarm = t
                warning_until = t + horizon + period
                warned.update(((run, str(i)) for i in hits))
                alarms.append(
                    {
                        "recording_id": run,
                        "time_sec": float(t),
                        "true_alarm": bool(hits),
                        "event_rows": [str(i) for i in hits],
                        "lead_times_sec": [float(selected.loc[i, "onset"] - t) for i in hits],
                    }
                )
                streak = 0
            warning_cells += int(t < warning_until)
            cell_end = t + exposure_step
            exposure_seconds += exposure_step
            warning_seconds += max(0.0, min(cell_end, warning_until) - t)
            last = t
        for idx, event in selected.iterrows():
            key = (run, str(idx))
            leads = [
                lead
                for alarm in alarms
                if alarm["recording_id"] == run
                for row, lead in zip(alarm["event_rows"], alarm["lead_times_sec"])
                if row == str(idx)
            ]
            event_records.append(
                {
                    "recording_id": run,
                    "event_row": str(idx),
                    "event_type": event.eventType,
                    "onset_sec": float(event.onset),
                    "eligible": key in eligible_events,
                    "warned": key in warned,
                    "first_warning_lead_sec": max(leads) if leads else None,
                }
            )
    hours = exposure_seconds / 3600
    false = sum((not a["true_alarm"] for a in alarms))
    leads = [e["first_warning_lead_sec"] for e in event_records if e["warned"]]
    return {
        "eligible_events": len(eligible_events),
        "warned_events": len(warned),
        "selected_events_in_evaluated_runs": len(event_records),
        "unevaluable_events_in_evaluated_runs": len(event_records) - len(eligible_events),
        "missed_eligible_events": len(eligible_events) - len(warned),
        "event_sensitivity": len(warned) / len(eligible_events) if eligible_events else None,
        "alarms": len(alarms),
        "false_alarms": false,
        "eligible_monitoring_hours": hours,
        "false_alarms_per_eligible_day": false / (hours / 24) if hours else None,
        "false_alarms_per_eligible_hour": false / hours if hours else None,
        "alarm_precision": (len(alarms) - false) / len(alarms) if alarms else None,
        "warning_hours_on_eligible_support": warning_seconds / 3600,
        "time_in_warning_fraction_eligible": (
            warning_seconds / exposure_seconds if exposure_seconds else None
        ),
        "median_first_warning_lead_sec": float(np.median(leads)) if leads else None,
        "fraction_eligible_decisions_in_warning": (
            warning_cells / len(data) if len(data) else None
        ),
        "evaluation_scope": "Only eligible forecast windows; excluded periods and run boundaries are not evaluated",
        "warning_definition": "[alarm time, alarm time + warning horizon + occurrence period), intersected with eligible decision cells",
        "event_records": event_records,
        "alarm_records": alarms,
    }


def choose_threshold(frame, probabilities, config, events_by_run):
    results = []
    for threshold in config["training"]["alarms"]["thresholds"]:
        metrics = alarm_metrics(frame, probabilities, threshold, config, events_by_run)
        results.append({"threshold": threshold, **metrics})
    budget = config["training"]["alarms"]["max_false_alarms_per_day"]
    feasible = [r for r in results if r["false_alarms_per_eligible_day"] <= budget]
    if not feasible:
        raise ValueError(
            "No validation alarm threshold meets the false-alarm budget; test set remains unopened"
        )
    best = max(
        feasible,
        key=lambda r: (
            r["event_sensitivity"] or 0,
            -r["false_alarms_per_eligible_day"],
            r["threshold"],
        ),
    )
    return (best, results)


In [ ]:
def classification_scores(labels, probabilities, threshold, training_prevalence):
    labels, probabilities = np.asarray(labels), np.asarray(probabilities, dtype=float)
    if (
        len(labels) != len(probabilities)
        or not np.isfinite(probabilities).all()
        or ((probabilities < 0) | (probabilities > 1)).any()
    ):
        raise ValueError("Invalid predicted probabilities.")
    tn, fp, fn, tp = confusion_matrix(
        labels, probabilities >= threshold, labels=[0, 1]
    ).ravel()
    both = len(np.unique(labels)) == 2
    reference = brier_score_loss(labels, np.full(len(labels), training_prevalence))
    brier = brier_score_loss(labels, probabilities)
    return {
        "windows": len(labels),
        "positive_windows": int(labels.sum()),
        "prevalence": float(labels.mean()),
        "average_precision": (
            float(average_precision_score(labels, probabilities)) if both else None
        ),
        "roc_auc": float(roc_auc_score(labels, probabilities)) if both else None,
        "brier_score": float(brier),
        "brier_skill_score": 1 - brier / reference if reference > 0 else None,
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
        "precision": float(tp / (tp + fp)) if tp + fp else None,
        "recall": float(tp / (tp + fn)) if tp + fn else None,
    }


def predict_metadata(model, session, rows, normalization, device):
    dataset = WaveformDataset(session["processed"], rows, normalization)
    try:
        loader = DataLoader(
            dataset,
            batch_size=session["config"]["network"]["batch_size"],
            shuffle=False,
            num_workers=0,
        )
        return predict_waveforms(model, loader, device)
    finally:
        dataset.close()


def finalize_report(session, partitions, model, normalization, device, records):
    """Validation selects threshold; test is predicted once; save tables and plain-language report."""
    cfg = session["config"]
    events = {r["id"]: pd.read_csv(r["events_file"], sep="\t") for r in records}
    validation = partitions["validation"]
    validation_probabilities = predict_metadata(
        model, session, validation, normalization, device
    )
    chosen, choices = choose_threshold(validation, validation_probabilities, cfg, events)
    threshold = chosen["threshold"]
    display(
        pd.DataFrame(choices)[
            ["threshold", "event_sensitivity", "false_alarms_per_eligible_day", "alarms"]
        ]
    )
    print("Fixed validation threshold:", threshold)
    # Mark immediately before any test prediction, so report failures do not reopen tuning.
    status(session, "evaluating_test", test_opened=True, threshold=threshold)
    test = partitions["test"]
    test_probabilities = predict_metadata(model, session, test, normalization, device)
    prevalence = float(partitions["train"].label.mean())
    results, prediction_tables = {}, []
    for name, rows, probabilities in [
        ("validation", validation, validation_probabilities),
        ("test", test, test_probabilities),
    ]:
        alarms = alarm_metrics(rows, probabilities, threshold, cfg, events)
        scores = classification_scores(
            rows.label.to_numpy(), probabilities, threshold, prevalence
        )
        results[name] = {"window": scores, "alarms": alarms}
        prediction_tables.append(rows.assign(partition=name, probability=probabilities))
    results["per_test_patient"] = {}
    for patient in sorted(test.subject.unique()):
        mask = (test.subject == patient).to_numpy()
        results["per_test_patient"][patient] = {
            "window": classification_scores(
                test.label.to_numpy()[mask], test_probabilities[mask], threshold, prevalence
            ),
            "alarms": alarm_metrics(
                test.loc[mask], test_probabilities[mask], threshold, cfg, events
            ),
        }
    checkpoint_path = session["model"] / "model.pt"
    checkpoint = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    checkpoint.update(
        threshold=float(threshold),
        training_prevalence=prevalence,
        patient_splits={
            name: sorted(rows.subject.unique()) for name, rows in partitions.items()
        },
    )
    torch.save(checkpoint, checkpoint_path)
    # Reconstruct from ordinary weights/config, without importing local implementation modules.
    restored = make_network(len(checkpoint["channels"]), checkpoint["config"]).to(device)
    restored.load_state_dict(checkpoint["state_dict"])
    restored_probabilities = predict_metadata(
        restored, session, validation, checkpoint["normalization"], device
    )
    np.testing.assert_allclose(
        restored_probabilities, validation_probabilities, rtol=1e-6, atol=1e-7
    )
    predictions = pd.concat(prediction_tables, ignore_index=True)
    predictions.to_csv(session["report"] / "predictions.csv", index=False)
    save_json(
        session["report"] / "metrics.json",
        {
            "threshold": threshold,
            "training_prevalence": prevalence,
            "threshold_candidates": choices,
            **results,
        },
    )
    table = pd.DataFrame(
        {
            name: result["window"]
            for name, result in results.items()
            if name in ["validation", "test"]
        }
    )
    lines = [
        "# EEG waveform CNN + LSTM",
        "",
        f"Session: {session['root'].name}",
        f"Channels in model order: {session['channels']}",
        f"Observed input: {cfg['preprocessing']['window_sec']} seconds at {session['output_fs']} Hz.",
        f"Forecast: {cfg['labels']['minimum_warning_min']} to {cfg['labels']['minimum_warning_min']+cfg['labels']['occurrence_period_min']} minutes ahead.",
        f"Seizure target: {cfg['labels'].get('seizure_group', 'custom')}; {cfg['labels']['seizure_prefixes']}; match={cfg['labels'].get('seizure_match', 'prefix')}",
        "Positive = selected future seizure; unrelated seizures are excluded, not negative training examples.",
        f"Fixed threshold: {threshold}",
        "",
        "Patient assignments:",
        json.dumps(checkpoint["patient_splits"], indent=2),
        "",
        "Window metrics:",
        table.to_string(),
        "",
        "Held-out test alarm metrics:",
        json.dumps(
            {
                k: v
                for k, v in results["test"]["alarms"].items()
                if k not in ["event_records", "alarm_records"]
            },
            indent=2,
        ),
        "",
        "Scores reflect eligible windows only. Sparse decisions and excluded intervals do not measure continuous clinical false-alarm burden.",
        "Validation was used for early stopping and threshold selection. Class-weighted risk scores are not independently calibrated.",
        "An idle/no-alarm operating point can meet the false-alarm budget while missing seizures; inspect event_sensitivity.",
        "More network depth or raw-waveform input does not establish improved forecasting. No patient-baseline adaptation is implemented.",
        "Saved-model reload validation predictions matched. All model inputs were filtered waveform samples, not handcrafted summary features.",
    ]
    (session["report"] / "report.md").write_text("\n".join(lines), encoding="utf-8")
    status(
        session,
        "complete",
        test_opened=True,
        threshold=float(threshold),
        saved_model_reload_matches=True,
    )
    return results, table


### Run: evaluate once and save the final model/report

**Output:** validation threshold choices, held-out metrics and three session folder paths. The final checkpoint contains weights, architecture settings, channel order, sample rate, training normalization and threshold. `report/session.json` reaches `complete` only after prediction reload verification. If it fails, inspect its recorded traceback. The test guard requires a new session for another experiment; do not tune against repeatedly viewed test scores.

In [ ]:
if SESSION["config"]["run_training"]:
    saved_status = json.loads((SESSION["report"] / "session.json").read_text())
    if test_evaluated or saved_status.get("test_opened", False):
        raise RuntimeError(
            "Test already opened for this session. Inspect its saved report or start a genuinely new experiment."
        )
    try:
        test_evaluated = True
        RESULTS, SCORE_TABLE = finalize_report(
            SESSION, PARTITIONS, MODEL, NORMALIZATION, MODEL_DEVICE, RECORDS
        )
        display(SCORE_TABLE)
        display(
            pd.DataFrame(
                {name: RESULTS[name]["alarms"] for name in ["validation", "test"]}
            ).loc[
                [
                    "eligible_events",
                    "warned_events",
                    "event_sensitivity",
                    "false_alarms_per_eligible_day",
                    "alarms",
                ]
            ]
        )
        print("SESSION COMPLETE")
        print("Processed:", SESSION["processed"])
        print("Model:", SESSION["model"] / "model.pt")
        print("Report:", SESSION["report"] / "report.md")
    except Exception as error:
        status(SESSION, "evaluation_failed", error)
        raise
else:
    print("PREPARATION-ONLY SESSION COMPLETE:", SESSION["root"])


## 9. Reopen a saved model later — example

This optional cell does nothing until you set `SAVED_SESSION`. Run the import and definition cells first; you do not need to create another session or retrain to load weights. It loads a trusted local checkpoint with PyTorch's weights-only loader and rebuilds the visible architecture. New inference waveforms must use the saved **channel order, filtering, sampling rate and training normalization**; sample tensors must have the same input shape. The checkpoint's threshold may be `None` if fitting completed but threshold/evaluation did not.

This example loads the model only; it does not re-evaluate the held-out test set. Run `predict_metadata` on deliberately selected prepared rows when performing a separate inference workflow.

In [ ]:
SAVED_SESSION = None  # Example: ROOT / "intermediate" / "20261017_120000_eeg_abc123"
if SAVED_SESSION is not None:
    loaded = torch.load(
        Path(SAVED_SESSION) / "model" / "model.pt", map_location="cpu", weights_only=True
    )
    LOADED_MODEL = make_network(len(loaded["channels"]), loaded["config"])
    LOADED_MODEL.load_state_dict(loaded["state_dict"])
    LOADED_MODEL.eval()
    print("Loaded channels:", loaded["channels"], "| Threshold:", loaded["threshold"])


## 10. Small debugging checks

These checks validate a label boundary/unknown future, causal filter waveform shape, and neural-network input/output dimensions. They are quick engineering checks, not a performance claim. `%debug` in a new cell immediately after an exception opens the interactive debugger (`p variable_name`, then `q` to leave). Inspect `report/session.json` for errors and the history CSV for learning behavior.

| Problem | Check |
|---|---|
| Missing channel | Inventory, exact spelling, compatible selected patients |
| No positive/eligible examples | Forecast horizon, annotation coverage, postictal exclusions and recording duration |
| Failed readiness | Distinct patients, both labels, unique eligible seizure counts |
| Poor scores | Test prevalence, event sensitivity/false alarms, training/validation learning curves, data quality and study design |
| RAM/GPU pressure | Smaller batch size, shorter input, fewer channels or smaller network |
| Disk space | Each new session owns its processed waveforms; delete whole unwanted sessions when no files are open |

The `source/` folder may retain original non-EEG files from your existing download, but this notebook never discovers or opens them. No documentation/audit folder is required or created. The only new folders are the three under each session.

In [ ]:
def run_debug_checks():
    cfg = copy.deepcopy(SESSION["config"])
    cfg["labels"].update(seizure_prefixes=["sz"], seizure_match="prefix", seizure_group="all")
    cfg["labels"].update(
        minimum_warning_min=1, occurrence_period_min=1, postictal_exclusion_min=1
    )
    events = pd.DataFrame([{"onset": 180.0, "duration": 10.0, "eventType": "sz_foc"}])
    ends = np.array([60.0, 65.0, 120.0, 125.0, 480.0, 485.0])
    labels, reasons, _ = label_windows(ends - 5, ends, events, 600, cfg["labels"])
    np.testing.assert_array_equal(labels, [0, 1, 1, -1, 0, -1])
    assert reasons[-1] == "unobserved_future"
    selected_cfg = {**cfg["labels"], **resolve_seizure_target("focal", [], "prefix")}
    mixed = pd.DataFrame(
        [
            {"onset": 180.0, "duration": 5.0, "eventType": "sz_foc_a_nm"},
            {"onset": 400.0, "duration": 5.0, "eventType": "sz_gen_m_tonic"},
        ]
    )
    selected_labels, _, _ = label_windows(
        np.array([60.0, 280.0]), np.array([65.0, 285.0]), mixed, 600.0, selected_cfg
    )
    np.testing.assert_array_equal(selected_labels, [1, -1])
    exact = resolve_seizure_target("custom", ["sz_foc_a_nm"], "exact")
    assert is_target_seizure("sz_foc_a_nm", exact)
    assert not is_target_seizure("sz_foc_a_nm_behavior", exact)
    # Test a known EEG waveform with the same configured cleaning steps.
    fs = (
        SESSION["output_fs"]
        if cfg["preprocessing"]["target_fs"] is None
        else max(256, SESSION["output_fs"])
    )
    p = cfg["preprocessing"]
    times = np.arange(sample_count(p["window_sec"] + p["filter_warmup_sec"], fs)) / fs
    cleaned = filter_window(20e-6 * np.sin(2 * np.pi * 10 * times)[None, :], fs, cfg)
    assert cleaned.shape == (1, sample_count(p["window_sec"], SESSION["output_fs"]))
    assert np.isfinite(cleaned).all()
    # Isolate random initialization so running checks cannot perturb later training seeds.
    with torch.random.fork_rng():
        net = make_network(len(SESSION["channels"]), cfg).eval()
        with torch.inference_mode():
            values = net(torch.zeros(2, len(SESSION["channels"]), cleaned.shape[-1]))
        assert values.shape == (2,) and torch.isfinite(values).all()
    return [
        "selected seizure types and exact matching",
        "forecast boundaries",
        "unknown future excluded",
        "filtered waveform shape",
        "CNN/LSTM tensor shape",
    ]


print("Debug checks passed:", run_debug_checks())
